In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [12]:
data = pd.read_csv('Churn_Modelling_India_v2.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
onehot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
model = Sequential([
    Dense(64, activation='relu',
          input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)    
])
model.summary()

c:\Users\Code and Debug\Desktop\AI ML Bootcamp Live Codes\19. End to End Deep Learning Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(optimizer='adam',
              loss='mean_absolute_error',
              metrics=['mae'])

In [16]:
log_dir = 'regressionlogs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10,
                                        restore_best_weights=True)

In [17]:
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test), epochs=1000,
                    callbacks=[tensorboard_callback, early_stopping_callback])

Epoch 1/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 730738.1875 - mae: 730738.1875 - val_loss: 731129.0625 - val_mae: 731129.0625
Epoch 2/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - loss: 730070.6250 - mae: 730070.6250 - val_loss: 729752.0625 - val_mae: 729752.0625
Epoch 3/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step - loss: 727580.6250 - mae: 727580.6250 - val_loss: 725994.9375 - val_mae: 725994.9375
Epoch 4/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 991us/step - loss: 722240.8125 - mae: 722240.8125 - val_loss: 718979.3125 - val_mae: 718979.3125
Epoch 5/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 951us/step - loss: 713285.3125 - mae: 713285.3125 - val_loss: 708043.1250 - val_mae: 708043.1250
Epoch 6/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 986us/step - loss: 700113.9375 - mae: 700113.9375 - val_loss: 692650.3125 - val_mae: 692650.3125
Epoch 7/1000
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 967us/step - loss: 682279.6250 - mae: 682279.6250 - val_loss: 672389.3750 - val_mae: 672389.3750
Epoch 8/

In [19]:
%load_ext tensorboard
%tensorboard --logdir regressionlogs/fit

In [20]:
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test MAE: {test_mae}')
# Is that good? Compare with a baseline
baseline_mae = (y_test - y_train.mean()).abs().mean()
print(f'Baseline MAE: {baseline_mae}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step - loss: 76048.7656 - mae: 76048.7656
Test MAE: 76048.765625
Baseline MAE: 121026.30474445375


In [21]:
model.save('regression_model.keras')

In [ ]:
# Save encoders and scaler for the regression model (separate from churn model files)
with open('regression_label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('regression_onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('regression_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [22]:
tf.keras.backend.clear_session()